# Module 7: Training
### BWSI SGAI 2026

By the end of this module you'll know:
- Where the hyperparameters live for both CNN and RL training
- How to run (or rerun) training from the command line
- How to run a controlled experiment without confounding your results
- Why the weights file might not update on a short run

---
*Open this notebook in JupyterLab. Press `Alt+R` to enter slideshow mode.*  
*Students: just scroll through normally and run the exercise cells.*

## How were the models trained?

The game ships with two pre-trained models in `models/weights/`:

| File | Model | What it does |
|------|-------|--------------|
| `baselineCNN.pth` | CNN classifier | Identifies humanoid type from an image |
| `baselineRL.pth` | RL policy | Decides which action to take each turn |

**You don't have to retrain** — the baselines work out of the box.  
But if you want to improve the models, change their behavior, or run an experiment, you need to know how training works.

This module covers **two separate training pipelines** — CNN and RL — and how to run each one.

## Where does training live?

```
bwsi-sgai/
├── gameplay/
├── models/
├── game_interfaces/
├── ui/
├── training/           <- this module
|   ├── train_cnn.py   <- now
|   ├── train_rl.py
|   ├── train_cnn.ipynb
|   └── train_rl.ipynb
└── tests/
```

*Both training scripts save weights to `models/weights/` — outside this directory.*  
*The notebooks are companion interfaces to the scripts — we'll come back to them.*

*`bwsi-sgai/training/train_cnn.py`*

## CNN Training — Locate

```
bwsi-sgai/
├── training/
|   ├── train_cnn.py   <- here
|   ├── train_rl.py
|   ├── train_cnn.ipynb
|   └── train_rl.ipynb
```

Open `training/train_cnn.py`. Structure:

```
train_cnn.py
├── Hyperparameters block    (lines 61-70)   <- start here
├── set_seed()               — reproducibility
├── HumanoidDataset          — PyTorch Dataset wrapping the image CSV
├── _save_cnn_plots()        — writes loss/accuracy curves and confusion matrix
└── train()                  — the full training loop
```

The file you care about is mostly the hyperparameter block. Everything else is supporting machinery.

*`bwsi-sgai/training/train_cnn.py`*

## CNN Training — Orient

The hyperparameter block is the first thing to read:

```python
# -- Hyperparameters ---------------------------------------------------
NUM_EPOCHS: int = 10       # training passes over the full dataset
BATCH_SIZE: int = 8        # images per gradient update
LEARNING_RATE: float = 0.001  # Adam optimizer step size
VAL_SPLIT: float = 0.2     # fraction of data held out for validation

SEED: int | None = 42
# ----------------------------------------------------------------------
```

**Important:** this dataset has about 200 images total. After `VAL_SPLIT=0.2`, only ~160 are used for training.  
On a dataset this small, overfitting typically sets in around 15-20 epochs — training accuracy keeps climbing while validation accuracy peaks and drops.  
**Keep `NUM_EPOCHS` under 20.** Watch the curves in the companion notebook to see when it happens.

**The interesting seam — augmentation:**

Inside `train()`, there are commented-out transforms:

```python
train_transforms = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomHorizontalFlip(),
    # transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    # ^ ColorJitter shifts brightness, contrast, saturation, and hue.
    # transforms.RandomRotation(degrees=15),
    # ^ RandomRotation tilts images up to +/-15 degrees.
    transforms.ToTensor(),
    _normalize,
])
```

The flip is active (mirroring is always safe). The others are experiments for you to try. Each comment explains what to think about before uncommenting.

*`bwsi-sgai/training/train_cnn.py`*

## CNN Training — Touch

Run the cell below to read the current hyperparameter values directly from the module.

In [1]:
import os
import sys
from pathlib import Path
%matplotlib inline

# Ensure the project root is the working directory and is on sys.path,
# so local packages (game_interfaces, gameplay, models) are importable
# and relative paths like DATA_DIR="data" resolve correctly.
# Works whether Jupyter was launched from bwsi-sgai/ or lectures/.
_here = Path().resolve()
_root = _here.parent if _here.name == "lectures" else _here
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [2]:
import training.train_cnn as tc

print(f"NUM_EPOCHS    = {tc.NUM_EPOCHS}")
print(f"BATCH_SIZE    = {tc.BATCH_SIZE}")
print(f"LEARNING_RATE = {tc.LEARNING_RATE}")
print(f"VAL_SPLIT     = {tc.VAL_SPLIT}")
print(f"SEED          = {tc.SEED}")
print(f"CNN_LOG_DIR   = {tc.CNN_LOG_DIR}")

NUM_EPOCHS    = 10
BATCH_SIZE    = 8
LEARNING_RATE = 0.001
VAL_SPLIT     = 0.2
SEED          = 42
CNN_LOG_DIR   = training/CNN-logs


*`bwsi-sgai/training/train_cnn.py`*

## Running CNN Training

**Two ways to train:**

```bash
# Via main.py (recommended — supports -e, -s, -d flags)
python main.py train --type cnn
python main.py train --type cnn -e 15 -s 7

# Direct script (works thanks to the sys.path fix at the top)
python training/train_cnn.py
```

**What happens:**
1. Loads all images from `data/`
2. Splits 80% train / 20% validation
3. Trains `DefaultCNN` for `NUM_EPOCHS` passes
4. Saves final weights to `models/weights/baselineCNN.pth`
5. Saves loss curves + confusion matrix to `training/CNN-logs/`

**Time estimate: ~2 minutes on CPU.** If it takes longer, kill it and use the pre-trained baseline.

> **Fallback:** If CNN training is taking too long, load `models/weights/baselineCNN.pth` directly and skip to the evaluation step:  
> ```python
> from models.CNNClassifier import CNNClassifier
> clf = CNNClassifier(img_data_root="data")
> # clf already loads weights from models/weights/baselineCNN.pth automatically
> ```
> The baseline is already there — training is only needed if you want to change something.

## Now: RL training

```
bwsi-sgai/
├── gameplay/
├── models/
├── game_interfaces/
├── ui/
├── training/
|   ├── train_cnn.py
|   ├── train_rl.py    <- now
|   ├── train_cnn.ipynb
|   └── train_rl.ipynb
└── tests/
```

*RL training is the main exercise target for this module.*  
*The CNN exercise is optional; the RL exercise is the one you'll run.*

*`bwsi-sgai/training/train_rl.py`*

## RL Training — Locate

Open `training/train_rl.py`. Structure:

```
train_rl.py
├── Hyperparameters block    (lines 75-89)   <- start here
├── set_seed()               — reproducibility
├── make_env()               — builds TrainEnv (game environment for training)
├── _save_rl_plots()         — writes reward curve + epsilon decay plots
└── train()                  — the full DQN training loop
```

The training loop runs the game `NUM_EPISODES` times, each time letting the agent play through a full shift. After each episode it updates the DQN weights using experience replay.

*`bwsi-sgai/training/train_rl.py`*

## RL Training — Orient

```python
# -----------------------------------------------------------------------
# Hyperparameters -- edit these to experiment
# -----------------------------------------------------------------------
NUM_EPISODES: int = 2500
BATCH_SIZE: int = 32
BUFFER_CAPACITY: int = 5000
EPSILON_DECAY: int = 1250  # episodes over which epsilon decays to epsilon_end
TARGET_UPDATE_FREQ: int = 20
GAMMA: float = 0.99
LEARNING_RATE: float = 1e-3
# -----------------------------------------------------------------------
```

Three constants students are most likely to change:

| Constant | Default | What it controls |
|----------|---------|------------------|
| `NUM_EPISODES` | 2500 | How long training runs (~20-40 min) |
| `EPSILON_DECAY` | 1250 | How fast the agent stops exploring |
| `GAMMA` | 0.99 | How much the agent values future rewards |

**GAMMA — the planning horizon:**

`GAMMA = 0.99` means the agent values a reward 10 turns from now at `0.99^10 ≈ 0.90` of its face value — nearly as much as an immediate reward.  
Lower values (e.g. `0.9`) make it more short-sighted: `0.9^10 ≈ 0.35`.  

A short-sighted agent might squish everything fast (cheap action) without thinking about future ambulance capacity. A far-sighted agent learns to manage capacity strategically.

**EPSILON_DECAY — exploration schedule:**

Early in training, the agent picks random actions (exploration). `EPSILON_DECAY = 1250` means it takes 1250 episodes to decay from fully random to mostly policy-driven. Raising this keeps exploration going longer — useful when your agent seems to get stuck early.

*`bwsi-sgai/training/train_rl.py`*

## RL Training — Touch

Run the cell below to confirm the current hyperparameter values directly from the module.

In [3]:
import training.train_rl as tr

print(f"NUM_EPISODES      = {tr.NUM_EPISODES}")
print(f"BATCH_SIZE        = {tr.BATCH_SIZE}")
print(f"BUFFER_CAPACITY   = {tr.BUFFER_CAPACITY}")
print(f"EPSILON_DECAY     = {tr.EPSILON_DECAY}")
print(f"TARGET_UPDATE_FREQ = {tr.TARGET_UPDATE_FREQ}")
print(f"GAMMA             = {tr.GAMMA}")
print(f"LEARNING_RATE     = {tr.LEARNING_RATE}")
print(f"SEED              = {tr.SEED}")
print(f"RL_LOG_DIR        = {tr.RL_LOG_DIR}")

NUM_EPISODES      = 2500
BATCH_SIZE        = 32
BUFFER_CAPACITY   = 5000
EPSILON_DECAY     = 1250
TARGET_UPDATE_FREQ = 20
GAMMA             = 0.99
LEARNING_RATE     = 0.001
SEED              = 42
RL_LOG_DIR        = training/RL-logs


*`bwsi-sgai/training/train_rl.py`*

## Running RL Training

**Full run (2500 episodes, ~20-40 min):**

```bash
python main.py train --type rl
```

**Fast baseline (500 episodes, ~5-10 min):**

```bash
python main.py train --type rl -e 500
```

**Direct script (same as fast baseline):**

```bash
python training/train_rl.py
```

The `-e` flag overrides `NUM_EPISODES` from the CLI without editing the file.  
Reward curves and epsilon decay plots are saved to `training/RL-logs/training_curves.png` after the run.

**Why might the weights file not update?**

The RL script only saves weights when two conditions are both true:

```python
if episode_reward > best_reward and agent.epsilon < 0.5:
    agent.save(save_path)
```

The `epsilon < 0.5` guard means: **no weights are saved while the agent is still mostly exploring.**  
With `EPSILON_DECAY = 1250`, epsilon reaches 0.5 around episode 625.  

If you run only 500 episodes, epsilon never drops below 0.5, and no weights are saved.  
The script will print: `"No weights saved — epsilon never dropped below 0.5."`

**Fix:** use at least 700-800 episodes when experimenting, or use the pre-trained baseline.

INSTRUCTOR NOTES:
Emphasize that epsilon < 0.5 guard is intentional — a lucky random episode early in training shouldn't overwrite the weights. A save at epsilon=0.9 would be mostly random luck, not a learned policy.
Ask students: why would it be bad to save weights from a high-epsilon episode even if the reward was high?
If students plan to run 500 episodes: adjust EPSILON_DECAY down to 250 in train_rl.py so epsilon reaches 0.5 by episode 250, giving the second half to actually learn and save.

## The Comparison Trap

When you run an experiment, you're comparing two conditions.  
**The only thing that should differ between conditions is your experimental variable.**

This is the most common mistake in RL experiments:

```
Control:    2500 episodes, GAMMA=0.99
Experiment:  500 episodes, GAMMA=0.90
```

What did you change? **Two things at once.** You changed both GAMMA *and* training time.  
Any difference in the results could be from either variable — you can't tell which.

**The correct comparison:**

```
Control:     500 episodes, GAMMA=0.99   <- same episodes
Experiment:  500 episodes, GAMMA=0.90   <- only GAMMA changed
```

Pick an episode count and **lock it for both conditions.**  
500 is a reasonable fast baseline for in-class experiments — just use it consistently.

INSTRUCTOR NOTES:
This is worth emphasizing before students run the exercise. The instinct is to compare their 500-episode run against the existing 2500-episode baseline — that comparison is always confounded.
Ask: if my 500-episode GAMMA=0.90 agent scores higher than the 2500-episode GAMMA=0.99 baseline, does that prove GAMMA=0.90 is better?
Answer: no — the 2500-episode agent had 5x more training. You're measuring training time, not your variable.
The clean interpretation: run both conditions at 500 episodes, compare those two numbers directly.

## Logs Append — Delete Before a New Experiment

When you run `python main.py -m auto --log`, results are **appended** to:

| File | Contents |
|------|----------|
| `log.csv` | One row per turn across all games |
| `summary.csv` | One row per completed game |

They never reset automatically. If you ran 5 games with the old baseline and 5 with your new agent, `summary.csv` will have 10 rows — you need to filter by timestamp or seed to separate them.

**Cleaner approach:** delete both files before starting a new experiment:

```bash
del log.csv summary.csv
```

Then re-run your evaluation games. You'll get a clean slate with only the rows you care about.

## Companion Notebooks

```
bwsi-sgai/
├── training/
|   ├── train_cnn.py
|   ├── train_rl.py
|   ├── train_cnn.ipynb    <- companion
|   └── train_rl.ipynb     <- companion
```

*`bwsi-sgai/training/train_rl.ipynb` · `bwsi-sgai/training/train_cnn.ipynb`*

## The Notebooks — Orient

Each notebook is an interactive version of the corresponding script. They show live loss curves, reward curves, and confusion matrices as training runs.

**When to use the notebook vs. the script:**

| Situation | Use |
|-----------|-----|
| Quick retrain, just want the weights | Script via CLI |
| Want to watch curves as it trains | Notebook |
| Running multiple conditions in a loop | Script via CLI |
| Debugging — why is accuracy not improving? | Notebook |

*`bwsi-sgai/training/train_rl.ipynb` · `bwsi-sgai/training/train_cnn.ipynb`*

## The Notebooks — A Critical Warning

The notebooks and their corresponding `.py` files **have separate hyperparameter copies.**

```
train_rl.ipynb:   NUM_EPISODES = 2500   <-- editing this...
train_rl.py:      NUM_EPISODES = 2500   <-- ...does NOT change this
```

Editing a constant in the notebook only affects that notebook's run.  
Running `python main.py train --type rl` always reads from `train_rl.py` — the notebook is irrelevant.

**Rule of thumb:**  
- Edit `train_rl.py` to change what the CLI and `main.py` use.  
- Edit the notebook to experiment interactively — but don't expect those changes to carry over.

The `.py` file is canonical. The notebook is a view into it, not a replacement.

*`bwsi-sgai/training/`*

## Notebooks — Touch

Run the cell below to confirm both notebooks exist in the training directory.

In [4]:
import os

notebooks = [f for f in os.listdir("training/") if f.endswith(".ipynb")]
scripts   = [f for f in os.listdir("training/") if f.endswith(".py")]

print("Notebooks:")
for nb in sorted(notebooks):
    print(f"  training/{nb}")

print("\nScripts:")
for sc in sorted(scripts):
    print(f"  training/{sc}")

Notebooks:
  training/train_cnn.ipynb
  training/train_rl.ipynb

Scripts:
  training/train_cnn.py
  training/train_rl.py


## Exercise: Retrain RL at 500 Episodes

**Goal:** retrain the RL agent with fewer episodes and compare its game score against the 2500-episode baseline.

**Before you start:**
- Open `training/train_rl.py` and note the current `NUM_EPISODES = 2500`
- You will NOT edit the file — use the `-e` flag to override episode count from the CLI
- Delete `log.csv` and `summary.csv` if they exist (avoids mixing old runs with new ones)

**Step 1 — Train at 500 episodes (~5-10 min):**

```bash
python main.py train --type rl -e 500
```

Watch the output. You'll see episode rewards printed every 50 episodes and a save notification when epsilon drops below 0.5.

> **If you see `"No weights saved — epsilon never dropped below 0.5."`:**  
> With `EPSILON_DECAY = 1250` and only 500 episodes, epsilon won't reach 0.5 in time.  
> To fix this for a 500-episode run: edit `EPSILON_DECAY` to `250` in `train_rl.py`, then rerun.  
> This compresses the exploration schedule so the agent has ~250 episodes left to exploit and save weights.

## Exercise: Evaluate and Compare

**Step 2 — Run the newly trained agent in auto mode:**

```bash
python main.py -m auto --log --seed 42
```

This runs one full game headlessly and appends results to `summary.csv`.

**Step 3 — Read the results:**

Run the cell below to read `summary.csv`.

In [5]:
import pandas as pd

df = pd.read_csv("summary.csv")
print(f"Total rows in summary.csv: {len(df)}")
print()
print(df.tail(5).to_string(index=False))

Total rows in summary.csv: 6

                              run_id  shift_length  capacity  num_turns  killed  saved  saved_value    reward  seed
079a6547-b8f6-4c81-b393-f6c185e500a3           720        10          6       0      0     0.000000  0.000000   NaN
5e170596-e026-412e-a0d4-634cc325032c           720        10         11       1      2     1.494915  1.247457   NaN
616d960b-403d-4387-8b3f-eee6ae130c9e           720        10          6       1      1     0.102397  0.051199   NaN
012491d0-c972-494a-af23-6ccf1a5572d5           720        10         31       1     15     6.811098 10.405549   NaN
9373e9a1-a416-4603-8c63-c64d0bb0d983           720        10          6       0      0     0.000000  0.000000   NaN


## Exercise: What to Compare

**Step 4 — Compare:**

The columns that matter: `saved`, `killed`, `saved_value`, `reward`.

Look at your 500-episode run vs. what the pre-trained 2500-episode baseline used to produce.

> **Comparison trap reminder:**  
> If you're comparing your 500-episode run against the 2500-episode baseline,  
> you're measuring **training time**, not your experimental variable.  
> A lower score just means less training — it says nothing about your hypothesis.

**The right comparison:** run the baseline again at 500 episodes too, then compare the two 500-episode runs.

**Step 5 — Optional: change one hyperparameter and retrain**

If you want to experiment with `GAMMA` instead:
1. Edit `GAMMA` in `train_rl.py` (try `0.90` instead of `0.99`)
2. Retrain at 500 episodes: `python main.py train --type rl -e 500`
3. Evaluate and compare — **keep `-e 500` for both conditions**

INSTRUCTOR NOTES:
Give students the heads-up about epsilon decay before they start — otherwise they'll retrain for 10 minutes and get no weights.
If time is short: skip the retrain entirely and just run the pre-trained agent several times with different seeds. The question "does score vary across seeds?" is its own interesting experiment.
For teams with a research question already: this is the moment to help them think about their independent variable. What are they changing? Can they hold everything else fixed?
The summary.csv columns: depending on the game version, column names may include seed, timestamp, saved, killed, saved_value, reward — have students check their actual column headers first.

## Pop Quiz — Navigation

These are navigation questions, not recall. Which file do you open?

---

**Q1.** You want to slow down how fast the RL agent stops exploring — you want it to keep trying random actions for longer before committing to a policy. Which constant do you change?

A) `GAMMA` in `training/train_rl.py`  
B) `EPSILON_DECAY` in `training/train_rl.py`  
C) `LEARNING_RATE` in `training/train_rl.py`  
D) `NUM_EPISODES` in `training/train_rl.py`

**Answer: B — `EPSILON_DECAY` in `training/train_rl.py`**  
`EPSILON_DECAY` controls over how many episodes epsilon decays from 1.0 to near 0.  
Higher value = slower decay = more exploration for longer.  
`GAMMA` affects how the agent values future rewards, not how much it explores.

## Pop Quiz — Navigation

**Q2.** You trained a CNN for 30 epochs and the results look worse than the 10-epoch baseline. The training accuracy is 98%, but validation accuracy dropped to 40%. What's happening, and where would you look to fix it?

A) The model is underfitting — increase `LEARNING_RATE` in `training/train_cnn.py`  
B) The model is overfitting — reduce `NUM_EPOCHS` in `training/train_cnn.py`  
C) The dataset is corrupt — recheck `data/consolidated_metadata.csv`  
D) The augmentation is too aggressive — remove `RandomHorizontalFlip` from `train_cnn.py`

**Answer: B — overfitting; reduce `NUM_EPOCHS` in `training/train_cnn.py`**  
98% train accuracy + 40% val accuracy = classic overfitting. The model memorized the training images instead of learning generalizable features.  
On a dataset this small (~200 images), overfitting typically sets in around 15-20 epochs. Keep `NUM_EPOCHS` under 20.  
You can also add regularization (weight decay, dropout) or use early stopping — see the comments in `train_cnn.py`.

## Pop Quiz — Navigation

**Q3.** You edited `NUM_EPISODES = 1000` in `train_rl.ipynb` and ran all the notebook cells. Then you ran `python main.py train --type rl` from the terminal. How many episodes does the CLI run?

A) 1000 — because you already changed the constant  
B) 2500 — because `main.py` reads from `train_rl.py`, not the notebook  
C) 0 — the notebook change breaks the script  
D) It depends on which one you ran last

**Answer: B — 2500, because `main.py` reads from `train_rl.py`**  
The notebook and the `.py` file have separate hyperparameter copies. Editing one does not affect the other.  
To change what the CLI uses, edit `training/train_rl.py` directly — or use the `-e` flag: `python main.py train --type rl -e 1000`.

## What You Now Know

- `training/train_cnn.py` — CNN hyperparameters live here; keep `NUM_EPOCHS` under 20 to avoid overfitting on this dataset
- `training/train_rl.py` — RL hyperparameters live here; `NUM_EPISODES`, `EPSILON_DECAY`, `GAMMA` are your main levers
- Weights save to `models/weights/` — `baselineCNN.pth` and `baselineRL.pth`
- RL weights only save when `epsilon < 0.5` — short runs may produce no weights file
- Notebooks and scripts have **separate** hyperparameter copies — edit the `.py` file for CLI runs
- `log.csv` and `summary.csv` append — delete them before a new experiment
- For valid comparisons: change **one thing**, hold episode count fixed for both conditions

---

**Next: Module 8 — Tests**  
How do you know the code actually works? We'll run the test suite, read the coverage report, and add an assertion to an existing test.